<a href="https://colab.research.google.com/github/Franncippi/Proyecto-mentoria-M09/blob/Fran/modelado_saldo_neto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Práctico 3 — Modelado (enfoque alternativo)

**Mentoría M09 — El impacto de la IA en la fuerza laboral**

Enfoque: clasificador binario sobre el **signo de `saldo_neto_empleo`**

---

## Qué hace esta notebook

A diferencia de `03_modelado.ipynb` (que evita `saldo_neto_empleo` por su fiabilidad baja y en su lugar clasifica `tasa_creacion` y `tasa_desplazamiento` por separado), esta notebook toma el enfoque más directo y literal de la Hipótesis 2: clasificar si el saldo neto de empleo de una empresa fue positivo o negativo.

**Limitación documentada, no resuelta por este enfoque:** `saldo_neto_empleo` tiene fiabilidad 0,33 (P2, §10.6.2) — dos tercios de su varianza es ruido de medición trimestral, no característica estable de la empresa. Por la fórmula de atenuación de la teoría clásica de los tests, la correlación máxima teóricamente alcanzable con este target está acotada en √0,33 ≈ 0,57. Un desempeño moderado acá (por ejemplo, accuracy 0,65-0,75) no sería un mal resultado — sería un resultado cerca de su techo teórico. Queda anotado para la sección de interpretación, no para descartar el enfoque.

---
# 1. Enfoque elegido

## 1.1 La pregunta

La Hipótesis 2 (P1) pregunta si es posible predecir la pérdida o creación de empleo a partir de las características de adopción de IA de una empresa. Este enfoque la responde de la forma más directa posible: un único clasificador binario sobre `saldo_neto_empleo > 0`.

## 1.2 Por qué un solo clasificador, y no dos

`saldo_neto_empleo = tasa_creacion − tasa_desplazamiento` ya está calculado en el dataset curado (P2, §7.1). En vez de modelar las dos tasas por separado, se modela directamente el signo del resultado neto — la lectura más literal de "¿esta empresa terminó ganando o perdiendo empleo neto?".

## 1.3 Predictores

Mismo catálogo `PREDICTORES` de P2 §11.2: estructura, adopción, operación, gobernanza y contexto país (numéricas), más industria, región, etapa de adopción, política de IA del país y comité de ética (categóricas) — excluyendo siempre las variables de rol *resultado* (serían circularidad, no predictores contemporáneos).

---
# 2. Preparación final

## 2.1 Carga de datos

Misma unidad de análisis que en `03_modelado.ipynb`: la tabla por empresa, no el panel (evita el group leakage por `company_id` anotado en el P2).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

SEMILLA = 42
np.random.seed(SEMILLA)

RUTA_LOCAL = Path("../data/processed/empresas.parquet")
if RUTA_LOCAL.exists():
    ruta_datos = RUTA_LOCAL
else:
    !git clone -b modelado-tp3 --depth 1 https://github.com/Franncippi/Proyecto-mentoria-M09.git repo_m09
    ruta_datos = Path("repo_m09/data/processed/empresas.parquet")

empresas = pd.read_parquet(ruta_datos)
print(empresas.shape)
print("nulos:", empresas.isna().sum().sum())

## 2.2 Definición del target

`saldo_neto_positivo = 1` si `saldo_neto_empleo > 0`, `0` en caso contrario. A diferencia del corte por mediana de `03_modelado.ipynb`, **acá el balance de clases no está garantizado por construcción** — depende de cuántas empresas reales quedaron de cada lado de cero, no de un punto de corte elegido para partir la muestra a la mitad. Se verifica antes de seguir, y si aparece desbalance se ajusta con `class_weight="balanced"` en la sección 3 en vez de ignorarlo.

También hay que decidir qué hacer con el caso exacto `saldo_neto_empleo == 0` (ni ganó ni perdió empleo neto) — se revisa cuántos casos son antes de asignarlos a un lado u otro.

In [ ]:
print("empresas con saldo_neto_empleo == 0 exacto:", (empresas.saldo_neto_empleo == 0).sum())

# Los ceros exactos se tratan como "no positivo" (0), ya que la pregunta es especificamente
# "hubo ganancia neta de empleo" y un saldo de exactamente 0 no lo es.
empresas["saldo_neto_positivo"] = (empresas["saldo_neto_empleo"] > 0).astype(int)

balance = empresas["saldo_neto_positivo"].value_counts(normalize=True).round(3)
print("\nbalance de clases:")
print(balance)
print(f"\nratio mayoria/minoria: {balance.max() / balance.min():.2f}")

*(Revisar el ratio mayoría/minoría antes de seguir: por debajo de ~1,5 se puede tratar como aproximadamente balanceado; por encima, conviene usar `class_weight="balanced"` en los modelos que lo permiten y no confiar solo en accuracy como métrica.)*

## 2.3 Catálogo de predictores

Mismos cinco bloques numéricos y el bloque categórico del catálogo `PREDICTORES` de P2 §11.2.

In [ ]:
PRED_ESTRUCTURA = ["num_employees", "annual_revenue_usd_millions", "company_age"]
PRED_ADOPCION   = ["ai_adoption_rate", "years_using_ai", "num_ai_tools_used",
                    "ai_projects_active", "ai_budget_percentage", "ai_training_hours",
                    "antiguedad_ia_relativa"]
PRED_OPERACION  = ["task_automation_rate", "remote_work_percentage"]
PRED_GOBERNANZA = ["ai_failure_rate", "regulatory_compliance_score", "ai_risk_management_score"]
PRED_PAIS       = ["gdp_per_capita", "internet_penetration", "digital_maturity_index",
                    "ai_patent_filings_2024", "ai_researchers_per_million"]
PRED_CATEG      = ["industry", "region", "ai_adoption_stage", "country_ai_policy", "ai_ethics_committee"]

NUM_PREDICTORES = PRED_ESTRUCTURA + PRED_ADOPCION + PRED_OPERACION + PRED_GOBERNANZA + PRED_PAIS
PREDICTORES     = NUM_PREDICTORES + PRED_CATEG

# Mismas dos variantes que en 03_modelado.ipynb, por las mismas razones ya documentadas ahi:
# el pais no discrimina entre empresas de un mismo pais, y task_automation_rate esta muy pegada
# al mecanismo de desplazamiento (r=0.91 con adopcion, P2 SS11.2).
PREDICTORES_SIN_PAIS = [c for c in PREDICTORES if c not in PRED_PAIS]
PREDICTORES_SIN_TASK = [c for c in PREDICTORES if c != "task_automation_rate"]

print(f"completo      : {len(PREDICTORES)} predictores")
print(f"sin_pais      : {len(PREDICTORES_SIN_PAIS)} predictores")
print(f"sin_task_auto : {len(PREDICTORES_SIN_TASK)} predictores")

## 2.4 Split y estrategia de validación

Igual que en `03_modelado.ipynb`: split 80/20 estratificado por el target, reservado para la evaluación final, y `StratifiedKFold` sobre el train para comparar configuraciones sin tocar el test.

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold

train_idx, test_idx = train_test_split(
    empresas.index, test_size=0.2, random_state=SEMILLA,
    stratify=empresas["saldo_neto_positivo"])

print(f"train: {len(train_idx)}   test: {len(test_idx)}")
print("balance en train:", empresas.loc[train_idx, "saldo_neto_positivo"].mean().round(3))
print("balance en test :", empresas.loc[test_idx, "saldo_neto_positivo"].mean().round(3))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

## 2.5 Preprocesamiento

`StandardScaler` para numéricas + `OneHotEncoder` para categóricas, dentro de un `Pipeline` para que el ajuste ocurra solo sobre el train de cada fold (evita leakage de escalado/codificación — ver `03_modelado.ipynb` §2.5.4 para el detalle de por qué).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def construir_preprocesador(predictores_num, predictores_cat):
    return ColumnTransformer([
        ("num", StandardScaler(), predictores_num),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), predictores_cat),
    ])

---
# 3. Implementación y evaluación

## 3.1 Comparación de modelos y conjuntos de predictores

3 modelos × 3 conjuntos de predictores, vía `StratifiedKFold` sobre el train. `class_weight="balanced"` en logística y SVM en caso de que 2.2 haya mostrado desbalance (no tiene costo si las clases ya están parejas — con clases balanceadas, ponderar por clase no cambia nada). Además de accuracy/ROC-AUC/F1, se agrega `balanced_accuracy` — promedia el recall de cada clase por separado, así que no se deja engañar por un modelo que simplemente prediga siempre la clase mayoritaria si hay desbalance.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate

MODELOS = {
    "Regresion logistica": LogisticRegression(max_iter=1000, random_state=SEMILLA, class_weight="balanced"),
    "SVM (kernel lineal)": SVC(kernel="linear", probability=True, random_state=SEMILLA, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=SEMILLA, n_jobs=-1,
                                              class_weight="balanced"),
}

CONJUNTOS_PREDICTORES = {
    "completo": PREDICTORES,
    "sin_pais": PREDICTORES_SIN_PAIS,
    "sin_task_auto": PREDICTORES_SIN_TASK,
}

X_train_full = empresas.loc[train_idx]
y_train = empresas.loc[train_idx, "saldo_neto_positivo"]

resultados_cv = []
for nombre_conjunto, predictores in CONJUNTOS_PREDICTORES.items():
    predictores_num = [c for c in predictores if c in NUM_PREDICTORES]
    predictores_cat = [c for c in predictores if c in PRED_CATEG]
    X = X_train_full[predictores]

    for nombre_modelo, modelo in MODELOS.items():
        pipe = Pipeline([
            ("prep", construir_preprocesador(predictores_num, predictores_cat)),
            ("clf", modelo),
        ])
        scores = cross_validate(pipe, X, y_train, cv=cv,
                                 scoring=["accuracy", "balanced_accuracy", "roc_auc", "f1"])
        resultados_cv.append({
            "predictores": nombre_conjunto,
            "modelo": nombre_modelo,
            "accuracy": scores["test_accuracy"].mean(),
            "balanced_accuracy": scores["test_balanced_accuracy"].mean(),
            "roc_auc": scores["test_roc_auc"].mean(),
            "f1": scores["test_f1"].mean(),
        })

tabla_cv = pd.DataFrame(resultados_cv).sort_values("roc_auc", ascending=False)
tabla_cv.round(3)

## 3.2 Evaluación final sobre el test set

Se refita la mejor configuración (según la tabla de arriba) sobre todo el train, y se evalúa **una sola vez** sobre el test reservado — la métrica que se reporta como desempeño real, sin haber sido usada para elegir nada.

In [ ]:
from sklearn.metrics import (confusion_matrix, classification_report,
                               RocCurveDisplay, ConfusionMatrixDisplay)

# Completar con la configuracion ganadora de la tabla anterior:
PREDICTORES_GANADOR = PREDICTORES_SIN_PAIS   # <- ajustar segun tabla_cv
MODELO_GANADOR = LogisticRegression(max_iter=1000, random_state=SEMILLA, class_weight="balanced")  # <- ajustar

predictores_num_g = [c for c in PREDICTORES_GANADOR if c in NUM_PREDICTORES]
predictores_cat_g = [c for c in PREDICTORES_GANADOR if c in PRED_CATEG]

pipe_final = Pipeline([
    ("prep", construir_preprocesador(predictores_num_g, predictores_cat_g)),
    ("clf", MODELO_GANADOR),
])
pipe_final.fit(X_train_full[PREDICTORES_GANADOR], y_train)

X_test = empresas.loc[test_idx, PREDICTORES_GANADOR]
y_test = empresas.loc[test_idx, "saldo_neto_positivo"]
y_pred = pipe_final.predict(X_test)

print(classification_report(y_test, y_pred, target_names=["saldo <= 0", "saldo > 0"]))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0],
                                          display_labels=["saldo <= 0", "saldo > 0"])
axes[0].set_title("Matriz de confusion (test)")
RocCurveDisplay.from_estimator(pipe_final, X_test, y_test, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "--", color="gray", label="azar")
axes[1].set_title("Curva ROC (test)")
plt.tight_layout()
plt.show()

## 3.3 Interpretación de coeficientes / importancia de variables

Igual lógica que en `03_modelado.ipynb`: coeficientes estandarizados si el ganador es la logística (log-odds directos), `feature_importances_` si es Random Forest. Si aparece una variable con coeficiente muy por encima del resto (como pasó con `ai_adoption_rate` en el enfoque de `03_modelado.ipynb`), conviene repetir ahí mismo el chequeo de sacar el bloque completo de variables correlacionadas antes de reportar la importancia individual como definitiva.

In [ ]:
nombres_feat = pipe_final.named_steps["prep"].get_feature_names_out()

if hasattr(pipe_final.named_steps["clf"], "coef_"):
    pesos = pd.Series(pipe_final.named_steps["clf"].coef_[0], index=nombres_feat)
    print("=== Top 10 coeficientes (por |valor|) ===")
    print(pesos.reindex(pesos.abs().sort_values(ascending=False).index).head(10).round(3))
elif hasattr(pipe_final.named_steps["clf"], "feature_importances_"):
    pesos = pd.Series(pipe_final.named_steps["clf"].feature_importances_, index=nombres_feat)
    print("=== Top 10 importancias ===")
    print(pesos.sort_values(ascending=False).head(10).round(3))